**Dr** **Michael**, to view this prototype in action, you will need to proceed as follows : a) Run all, b) Navigate to the bottom of Cell 3, c) Choose notebook-please use test notebook I have provided for first run (Clicking Choose File will allow you to browse your files), d) Choose language for translation by entering the two letter code from the list provided(you will need to scroll left to see list), and press enter, and e) When asked Generate MP4 now y/n? Enter y in input box. When you press enter, the programme will execute, generating an MP4 available in 'files' to your left, and automatically downloading a copy to you.

# PeerReview_Prototype_2.0 — Auto-Scroll Code Cell Fix

This notebook keeps the existing upload / language / generate-MP4 flow, and now adds an auto-scroll fix for longer code cells so the active line stays visible while typing.

- code cells are replayed in **their original in-cell order**
- `##script`, `#comments`, and Python code are handled **line by line**
- comments and code appear in the **same notebook-style code cell**
- the exported video looks more like a **notebook playback** than separate grouped blocks

Current priority in this build:
- fix notebook-style visuals
- fix in-order playback inside code cells

ElevenLabs can still be added back later; this version keeps the existing fallback to gTTS.


**CELL 1: INSTALL AND IMPORTS**

In [8]:

# =============================================
# Cell 1: Install and imports
# =============================================

!pip -q install nbformat nbclient langdetect deep-translator moviepy pillow pygments gTTS elevenlabs requests

import os
import io
import re
import gc
import sys
import math
import json
import uuid
import base64
import shutil
import textwrap
import tempfile
import contextlib
from pathlib import Path

import nbformat

from PIL import Image, ImageDraw, ImageFont
from pygments import lex
from pygments.lexers import PythonLexer
from pygments.token import Token
from pygments.styles import get_style_by_name

from langdetect import detect, DetectorFactory
from deep_translator import GoogleTranslator
from gtts import gTTS
import requests

from moviepy.editor import (
    ImageClip,
    AudioFileClip,
    concatenate_videoclips,
)

from nbclient import NotebookClient
from nbclient.exceptions import CellExecutionError

try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    IN_COLAB = False

DetectorFactory.seed = 0

print("Setup complete.")
print(f"Running in Colab: {IN_COLAB}")


Setup complete.
Running in Colab: True


**CELL2: CONFIGURATION**

In [9]:

# =============================================
# Cell 2: Configuration
# =============================================

# -------------------------------
# Paths and working folders
# -------------------------------
WORK_DIR = Path(tempfile.mkdtemp(prefix="end2end_v2_"))
AUDIO_DIR = WORK_DIR / "audio"
FRAME_DIR = WORK_DIR / "frames"
RENDER_DIR = WORK_DIR / "render"
AUDIO_DIR.mkdir(parents=True, exist_ok=True)
FRAME_DIR.mkdir(parents=True, exist_ok=True)
RENDER_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_MP4_PATH = Path("/content/PeerReview_Prototype_2_0_Output.mp4" if IN_COLAB else str(Path.cwd() / "PeerReview_Prototype_2_0_Output.mp4"))

# -------------------------------
# Video settings
# -------------------------------
VIDEO_WIDTH = 1280
VIDEO_HEIGHT = 720
FPS = 6
BACKGROUND = (248, 250, 252)
NOTEBOOK_BG = (255, 255, 255)
NOTEBOOK_BORDER = (218, 223, 230)
MARKDOWN_BG = (250, 250, 252)
COMMENT_BG = (247, 250, 255)
OUTPUT_BG = (248, 250, 252)
SCRIPT_BG = (255, 255, 255)
HEADER_BG = (245, 247, 250)

LEFT = 48
RIGHT = 48
TOP = 48
INNER_PAD = 22
CELL_GAP = 18

CODE_TYPING_CPS = 22     # characters per second
MIN_CLIP_SECONDS = 1.1
PAUSE_AFTER_MARKDOWN = 0.6
PAUSE_AFTER_COMMENTS = 0.7
PAUSE_AFTER_OUTPUT = 1.0
SCRIPT_HOLD_SECONDS = 0.2
CODE_SCROLL_BOTTOM_PADDING_LINES = 2
CODE_SCROLLBAR_WIDTH = 8
CODE_SCROLLBAR_MIN_HEIGHT = 48

# -------------------------------
# Fonts
# -------------------------------
FONT_CANDIDATES_MONO = [
    "/usr/share/fonts/truetype/dejavu/DejaVuSansMono.ttf",
    "/usr/share/fonts/truetype/liberation2/LiberationMono-Regular.ttf",
]
FONT_CANDIDATES_SANS = [
    "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
    "/usr/share/fonts/truetype/liberation2/LiberationSans-Regular.ttf",
]

def load_font(candidates, size):
    for path in candidates:
        if os.path.exists(path):
            return ImageFont.truetype(path, size=size)
    return ImageFont.load_default()

FONT_SANS_20 = load_font(FONT_CANDIDATES_SANS, 20)
FONT_SANS_24 = load_font(FONT_CANDIDATES_SANS, 24)
FONT_SANS_28 = load_font(FONT_CANDIDATES_SANS, 28)
FONT_SANS_34 = load_font(FONT_CANDIDATES_SANS, 34)
FONT_MONO_22 = load_font(FONT_CANDIDATES_MONO, 22)

# -------------------------------
# Syntax highlighting
# -------------------------------
PYGMENTS_STYLE = get_style_by_name("default")
STYLE_MAP = {}
for token, style in PYGMENTS_STYLE.styles.items():
    color = None
    if style:
        match = re.search(r"(?<!bg:)([0-9a-fA-F]{6})", style)
        if match:
            hex_color = match.group(1)
            color = tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4))
    STYLE_MAP[token] = color or (34, 40, 49)

TOKEN_COLOR_FALLBACKS = {
    Token.Keyword: (0, 0, 180),
    Token.Name.Function: (121, 94, 38),
    Token.Literal.String: (3, 122, 41),
    Token.Comment: (120, 120, 120),
    Token.Operator: (32, 33, 36),
    Token.Name.Builtin: (111, 66, 193),
}

def token_color(tok):
    current = tok
    while current not in STYLE_MAP and current.parent is not None:
        current = current.parent
    if current in STYLE_MAP:
        return STYLE_MAP[current]
    for k, v in TOKEN_COLOR_FALLBACKS.items():
        if tok in k:
            return v
    return (34, 40, 49)

# -------------------------------
# Languages
# -------------------------------
LANGUAGE_OPTIONS = {
    "ar": "Arabic",
    "bn": "Bengali",
    "bg": "Bulgarian",
    "zh-CN": "Chinese (Simplified)",
    "zh-TW": "Chinese (Traditional)",
    "cs": "Czech",
    "da": "Danish",
    "nl": "Dutch",
    "en": "English",
    "fi": "Finnish",
    "fr": "French",
    "de": "German",
    "el": "Greek",
    "he": "Hebrew",
    "hi": "Hindi",
    "hu": "Hungarian",
    "id": "Indonesian",
    "it": "Italian",
    "ja": "Japanese",
    "ko": "Korean",
    "ms": "Malay",
    "no": "Norwegian",
    "fa": "Persian",
    "pl": "Polish",
    "pt": "Portuguese",
    "ro": "Romanian",
    "ru": "Russian",
    "sk": "Slovak",
    "es": "Spanish",
    "sv": "Swedish",
    "ta": "Tamil",
    "th": "Thai",
    "tr": "Turkish",
    "uk": "Ukrainian",
    "ur": "Urdu",
    "vi": "Vietnamese",
}

LANGUAGE_NAME_TO_CODE = {v.lower(): k for k, v in LANGUAGE_OPTIONS.items()}
LANGUAGE_NAME_TO_CODE.update({
    "mandarin": "zh-CN",
    "chinese": "zh-CN",
    "simplified chinese": "zh-CN",
})

# -------------------------------
# ElevenLabs defaults
# -------------------------------
ELEVENLABS_API_KEY = os.getenv("ELEVENLABS_API_KEY", "").strip()
ELEVENLABS_VOICE_ID = os.getenv("ELEVENLABS_VOICE_ID", "JBFqnCBsd6RMkjVDRZzb")
ELEVENLABS_MODEL_ID = os.getenv("ELEVENLABS_MODEL_ID", "eleven_multilingual_v2")

print(f"Working directory: {WORK_DIR}")
print("Configuration loaded.")


Working directory: /tmp/end2end_v2_n9xdfvhz
Configuration loaded.


**CELL 3: UPLOAD NOTEBOOK AND CHOOSE TRANSLATION LANGUAGE**

In [10]:

# ========================================================
# Cell 3: Upload notebook and choose translation language
# ========================================================


def upload_single_notebook_bytes():
    if not IN_COLAB:
        raise RuntimeError("This upload helper expects Google Colab.")
    print("Please choose the scripted notebook you want to convert to MP4.")
    uploaded = files.upload()
    if not uploaded:
        raise ValueError("No notebook was uploaded.")
    name = next(iter(uploaded.keys()))
    content = uploaded[name]
    if not name.endswith(".ipynb"):
        raise ValueError(f"Uploaded file is not an .ipynb notebook: {name}")
    return name, content

def normalise_text(text):
    text = text.replace("\u00a0", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text

def strip_markdown_symbols(text):
    text = re.sub(r"`{1,3}", "", text)
    text = re.sub(r"^\s{0,3}(#{1,6}|\*|-|\+|\d+\.)\s*", "", text, flags=re.M)
    text = re.sub(r"\[(.*?)\]\((.*?)\)", r"\1", text)
    return normalise_text(text)

def parse_code_annotations(code_text):
    scripts = []
    comments = []
    visible_code_lines = []
    for raw_line in code_text.splitlines():
        line = raw_line.rstrip("\n")
        stripped = line.strip()

        if stripped.startswith("##"):
            script_text = stripped[2:].strip()
            if script_text:
                scripts.append(script_text)
            continue

        if stripped.startswith("#"):
            comment_text = stripped[1:].strip()
            if comment_text:
                comments.append(comment_text)
            continue

        if "#" in line:
            code_part, comment_part = line.split("#", 1)
            visible_code_lines.append(code_part.rstrip())
            inline_comment = comment_part.strip()
            if inline_comment:
                comments.append(inline_comment)
        else:
            visible_code_lines.append(line)

    return {
        "scripts": scripts,
        "comments": comments,
        "visible_code_lines": visible_code_lines,
    }

def build_detection_corpus(nb):
    chunks = []
    for cell in nb.cells:
        if cell.cell_type == "markdown":
            cleaned = strip_markdown_symbols(cell.source)
            if cleaned:
                chunks.append(cleaned)
        elif cell.cell_type == "code":
            parsed = parse_code_annotations(cell.source)
            chunks.extend([s for s in parsed["scripts"] if s.strip()])
            chunks.extend([c for c in parsed["comments"] if c.strip()])
    joined = " ".join(chunks)
    return normalise_text(joined)

def detect_language_code(text):
    if not text or len(text) < 8:
        return "en"
    try:
        code = detect(text)
        if code == "zh-cn":
            return "zh-CN"
        if code == "zh-tw":
            return "zh-TW"
        return code
    except Exception:
        return "en"

def choose_target_language(source_lang_code):
    source_name = LANGUAGE_OPTIONS.get(source_lang_code, source_lang_code)
    print()
    print(f"Detected source language: {source_name} ({source_lang_code})")
    print("Available target languages:")
    for i, (code, name) in enumerate(LANGUAGE_OPTIONS.items(), start=1):
        print(f"{i:>2}. {name} ({code})")

    while True:
        answer = input("\nEnter target language code or full name (for example: fr or French): ").strip()
        answer_lower = answer.lower()
        if answer in LANGUAGE_OPTIONS:
            return answer
        if answer_lower in LANGUAGE_NAME_TO_CODE:
            return LANGUAGE_NAME_TO_CODE[answer_lower]
        print("Language not recognised. Please try again.")

def confirm_generation():
    while True:
        answer = input("Generate MP4 now? (y/n): ").strip().lower()
        if answer in {"y", "yes"}:
            return True
        if answer in {"n", "no"}:
            return False
        print("Please enter y or n.")

uploaded_name, uploaded_bytes = upload_single_notebook_bytes()
input_notebook_path = WORK_DIR / uploaded_name
input_notebook_path.write_bytes(uploaded_bytes)

ORIGINAL_NOTEBOOK = nbformat.read(io.BytesIO(uploaded_bytes), as_version=4)
DETECTION_CORPUS = build_detection_corpus(ORIGINAL_NOTEBOOK)
SOURCE_LANGUAGE_CODE = detect_language_code(DETECTION_CORPUS)
SOURCE_LANGUAGE_NAME = LANGUAGE_OPTIONS.get(SOURCE_LANGUAGE_CODE, SOURCE_LANGUAGE_CODE)

TARGET_LANGUAGE_CODE = choose_target_language(SOURCE_LANGUAGE_CODE)
TARGET_LANGUAGE_NAME = LANGUAGE_OPTIONS.get(TARGET_LANGUAGE_CODE, TARGET_LANGUAGE_CODE)

print()
print(f"Input notebook: {uploaded_name}")
print(f"Source language: {SOURCE_LANGUAGE_NAME} ({SOURCE_LANGUAGE_CODE})")
print(f"Target language: {TARGET_LANGUAGE_NAME} ({TARGET_LANGUAGE_CODE})")

GENERATE_MP4 = confirm_generation()
if not GENERATE_MP4:
    raise SystemExit("MP4 generation cancelled by user.")

print("Confirmed. Proceeding to notebook execution, translation, narration, and video export.")


Please choose the scripted notebook you want to convert to MP4.


Saving TTS_MP4PoC2.ipynb to TTS_MP4PoC2.ipynb

Detected source language: English (en)
Available target languages:
 1. Arabic (ar)
 2. Bengali (bn)
 3. Bulgarian (bg)
 4. Chinese (Simplified) (zh-CN)
 5. Chinese (Traditional) (zh-TW)
 6. Czech (cs)
 7. Danish (da)
 8. Dutch (nl)
 9. English (en)
10. Finnish (fi)
11. French (fr)
12. German (de)
13. Greek (el)
14. Hebrew (he)
15. Hindi (hi)
16. Hungarian (hu)
17. Indonesian (id)
18. Italian (it)
19. Japanese (ja)
20. Korean (ko)
21. Malay (ms)
22. Norwegian (no)
23. Persian (fa)
24. Polish (pl)
25. Portuguese (pt)
26. Romanian (ro)
27. Russian (ru)
28. Slovak (sk)
29. Spanish (es)
30. Swedish (sv)
31. Tamil (ta)
32. Thai (th)
33. Turkish (tr)
34. Ukrainian (uk)
35. Urdu (ur)
36. Vietnamese (vi)

Enter target language code or full name (for example: fr or French): en

Input notebook: TTS_MP4PoC2.ipynb
Source language: English (en)
Target language: English (en)
Generate MP4 now? (y/n): y
Confirmed. Proceeding to notebook execution, translat

# **INPUTS ARE IN ABOVE CELL**

**CELL 4: TRANSLATION, EXECUTION, AND EXTRACTION HELPERS**

In [11]:
# ======================================================
# Cell 4: Translation, execution, and extraction helpers
# ======================================================

TRANSLATION_CACHE = {}

def safe_translate(text, source_lang, target_lang):
    text = (text or "").strip()
    if not text:
        return ""
    if target_lang == source_lang:
        return text

    cache_key = (text, source_lang, target_lang)
    if cache_key in TRANSLATION_CACHE:
        return TRANSLATION_CACHE[cache_key]

    try:
        translated = GoogleTranslator(source="auto", target=target_lang).translate(text)
        translated = translated if translated else text
    except Exception:
        translated = text

    TRANSLATION_CACHE[cache_key] = translated
    return translated

def notebook_execution_copy(src_path):
    temp_nb_path = RENDER_DIR / f"exec_copy_{uuid.uuid4().hex}.ipynb"
    shutil.copy2(src_path, temp_nb_path)
    return temp_nb_path

def execute_notebook(path_to_notebook):
    exec_path = notebook_execution_copy(path_to_notebook)
    nb = nbformat.read(exec_path, as_version=4)

    client = NotebookClient(
        nb,
        timeout=600,
        kernel_name="python3",
        resources={"metadata": {"path": str(exec_path.parent)}},
        allow_errors=True,
    )

    try:
        executed_nb = client.execute()
    except CellExecutionError as exc:
        print("Notebook execution raised an error, but captured outputs will still be used where possible.")
        print(exc)
        executed_nb = client.nb

    executed_path = exec_path.parent / f"executed_{exec_path.name}"
    nbformat.write(executed_nb, executed_path)
    return executed_nb, executed_path

def outputs_to_renderables(outputs):
    renderables = []
    for out in outputs:
        out_type = out.get("output_type")

        if out_type == "stream":
            text = out.get("text", "")
            if isinstance(text, list):
                text = "".join(text)
            renderables.append({"kind": "text", "text": text})

        elif out_type in {"execute_result", "display_data"}:
            data = out.get("data", {})
            if "image/png" in data:
                renderables.append({"kind": "image", "image_b64": data["image/png"]})
            elif "text/plain" in data:
                text = data["text/plain"]
                if isinstance(text, list):
                    text = "".join(text)
                renderables.append({"kind": "text", "text": text})

        elif out_type == "error":
            traceback_lines = out.get("traceback", [])
            text = "\n".join(traceback_lines)
            renderables.append({"kind": "text", "text": text})

    return renderables

def parse_code_events_linewise(code_text, source_lang, target_lang):
    """
    Preserve original order inside each code cell.
    - ##script   -> narrated only
    - #comment   -> printed only
    - code line  -> typed on screen
    """
    events = []

    for raw_line in code_text.splitlines():
        line = raw_line.rstrip("\n")
        stripped = line.strip()

        if stripped.startswith("##"):
            script_text = stripped[2:].strip()
            translated = safe_translate(script_text, source_lang, target_lang) if script_text else ""
            if translated:
                events.append({
                    "kind": "script",
                    "text": translated,
                })

        elif stripped.startswith("#"):
            comment_text = stripped[1:].strip()
            translated = safe_translate(comment_text, source_lang, target_lang) if comment_text else ""
            events.append({
                "kind": "comment",
                "text": translated,
            })

        else:
            events.append({
                "kind": "code",
                "text": line,
            })

    return events

def build_presentation_plan(original_nb, executed_nb, source_lang, target_lang):
    plan = []
    executed_code_cells = [c for c in executed_nb.cells if c.cell_type == "code"]
    executed_code_idx = 0
    code_cell_number = 0

    for source_cell_index, cell in enumerate(original_nb.cells, start=1):
        if cell.cell_type == "markdown":
            original_text = strip_markdown_symbols(cell.source)
            translated = safe_translate(original_text, source_lang, target_lang)
            plan.append({
                "cell_type": "markdown",
                "source_cell_index": source_cell_index,
                "translated_text": translated,
            })

        elif cell.cell_type == "code":
            code_cell_number += 1
            outputs = []
            if executed_code_idx < len(executed_code_cells):
                outputs = outputs_to_renderables(executed_code_cells[executed_code_idx].get("outputs", []))
            executed_code_idx += 1

            events = parse_code_events_linewise(cell.source, source_lang, target_lang)

            plan.append({
                "cell_type": "code",
                "source_cell_index": source_cell_index,
                "code_cell_number": code_cell_number,
                "events": events,
                "outputs": outputs,
            })

    return plan

EXECUTED_NOTEBOOK, EXECUTED_NOTEBOOK_PATH = execute_notebook(input_notebook_path)
PRESENTATION_PLAN = build_presentation_plan(
    ORIGINAL_NOTEBOOK,
    EXECUTED_NOTEBOOK,
    SOURCE_LANGUAGE_CODE,
    TARGET_LANGUAGE_CODE,
)

print(f"Presentation plan created with {len(PRESENTATION_PLAN)} items.")


Presentation plan created with 10 items.


**CELL 5: RENDERING AND AUDIO HELPERS**

In [12]:
# ===================================
# Cell 5: Rendering and audio helpers
# ===================================

def make_canvas():
    return Image.new("RGB", (VIDEO_WIDTH, VIDEO_HEIGHT), BACKGROUND)

def draw_rounded_rect(draw, box, fill, outline=None, radius=18, width=1):
    draw.rounded_rectangle(box, radius=radius, fill=fill, outline=outline, width=width)

def wrap_text(draw, text, font, max_width):
    if text == "":
        return [""]
    words = text.split()
    if not words:
        return [""]
    lines = []
    current_line = words[0]
    for word in words[1:]:
        trial_line = current_line + " " + word
        if draw.textlength(trial_line, font=font) <= max_width:
            current_line = trial_line
        else:
            lines.append(current_line)
            current_line = word
    lines.append(current_line)
    return lines

def draw_wrapped_text(draw, text, x, y, font, fill, max_width, line_gap=10):
    lines = []
    for paragraph in str(text).split("\n"):
        if paragraph.strip():
            lines.extend(wrap_text(draw, paragraph, font, max_width))
        else:
            lines.append("")
    cursor_y = y
    for line in lines:
        draw.text((x, cursor_y), line, font=font, fill=fill)
        bbox = draw.textbbox((x, cursor_y), line if line else "Ag", font=font)
        cursor_y += (bbox[3] - bbox[1]) + line_gap
    return cursor_y

def paste_output_image(base_img, image_b64, box):
    try:
        raw = base64.b64decode(image_b64)
        output_img = Image.open(io.BytesIO(raw)).convert("RGB")
        output_img.thumbnail((box[2] - box[0], box[3] - box[1]))
        x = box[0] + ((box[2] - box[0]) - output_img.width) // 2
        y = box[1] + ((box[3] - box[1]) - output_img.height) // 2
        base_img.paste(output_img, (x, y))
    except Exception:
        pass

def select_code_window(all_lines, max_visible_lines, bottom_padding_lines=0):
    lines = list(all_lines or [])
    if max_visible_lines <= 0 or not lines:
        return [], 0

    bottom_padding_lines = max(0, min(bottom_padding_lines, max_visible_lines - 1))
    effective_capacity = max(1, max_visible_lines - bottom_padding_lines)

    if len(lines) <= effective_capacity:
        return lines, 0

    start_idx = len(lines) - effective_capacity
    return lines[start_idx:], start_idx

def draw_scrollbar(draw, track_box, start_idx, visible_count, total_count):
    if total_count <= visible_count or visible_count <= 0:
        return

    x0, y0, x1, y1 = track_box
    track_height = max(1, y1 - y0)
    thumb_height = max(
        CODE_SCROLLBAR_MIN_HEIGHT,
        int(track_height * (visible_count / total_count)),
    )
    thumb_height = min(thumb_height, track_height)

    max_start = max(1, total_count - visible_count)
    ratio = start_idx / max_start
    thumb_y = y0 + int((track_height - thumb_height) * ratio)

    draw.rounded_rectangle(track_box, radius=CODE_SCROLLBAR_WIDTH // 2, fill=(237, 241, 246))
    draw.rounded_rectangle((x0, thumb_y, x1, thumb_y + thumb_height), radius=CODE_SCROLLBAR_WIDTH // 2, fill=(194, 201, 211))

def render_notebook_frame(
    cell_kind="code",
    cell_number=1,
    markdown_text=None,
    display_lines=None,
    output_items=None,
    narration_active=False,
    typing_current=None,
    typing_cursor=False,
):
    img = make_canvas()
    draw = ImageDraw.Draw(img)

    # Outer notebook page
    page_box = (32, 28, VIDEO_WIDTH - 32, VIDEO_HEIGHT - 28)
    draw_rounded_rect(draw, page_box, fill=NOTEBOOK_BG, outline=NOTEBOOK_BORDER, radius=24, width=2)

    # Toolbar
    toolbar_box = (33, 29, VIDEO_WIDTH - 33, 84)
    draw_rounded_rect(draw, toolbar_box, fill=HEADER_BG, outline=None, radius=24, width=1)

    traffic_colours = [(255, 95, 86), (255, 189, 46), (39, 201, 63)]
    for i, colour in enumerate(traffic_colours):
        x0 = 54 + i * 18
        draw.ellipse((x0, 46, x0 + 10, 56), fill=colour)

    draw.text((122, 42), "Google Colab-style Notebook Preview", font=FONT_SANS_24, fill=(58, 64, 72))

    content_top = 104
    prompt_left = 50
    cell_left = 145
    cell_right = VIDEO_WIDTH - 52

    if cell_kind == "markdown":
        draw.text((prompt_left, content_top + 18), "Text", font=FONT_MONO_22, fill=(110, 116, 124))
        cell_box = (cell_left, content_top, cell_right, VIDEO_HEIGHT - 54)
        draw_rounded_rect(draw, cell_box, fill=MARKDOWN_BG, outline=NOTEBOOK_BORDER, radius=18, width=1)
        draw_wrapped_text(
            draw,
            markdown_text or "",
            cell_box[0] + 20,
            cell_box[1] + 20,
            FONT_SANS_28,
            (33, 37, 41),
            max_width=(cell_box[2] - cell_box[0] - 40),
            line_gap=10,
        )
        return img

    # Code cell
    draw.text((prompt_left, content_top + 18), f"In [{cell_number}]:", font=FONT_MONO_22, fill=(110, 116, 124))
    if narration_active:
        draw.text((cell_right - 220, content_top + 18), "Narration playing", font=FONT_SANS_20, fill=(88, 102, 135))

    cell_box = (cell_left, content_top, cell_right, VIDEO_HEIGHT - 54)
    draw_rounded_rect(draw, cell_box, fill=(255, 255, 255), outline=NOTEBOOK_BORDER, radius=18, width=1)

    x0 = cell_box[0] + 16
    y0 = cell_box[1] + 14
    line_h = 28
    lexer = PythonLexer()

    all_visible_lines = list(display_lines or [])
    if typing_current is not None:
        all_visible_lines = all_visible_lines + [{"kind": "code", "text": typing_current}]

    available_bottom = cell_box[3] - 12
    if output_items:
        available_bottom = cell_box[1] + 300

    max_visible_lines = max(1, (available_bottom - y0) // line_h)
    visible_lines, start_idx = select_code_window(
        all_visible_lines,
        max_visible_lines=max_visible_lines,
        bottom_padding_lines=CODE_SCROLL_BOTTOM_PADDING_LINES,
    )

    for line_idx, item in enumerate(visible_lines):
        y = y0 + line_idx * line_h
        if y + line_h > available_bottom:
            break

        line_kind = item.get("kind")
        line_text = item.get("text", "")

        if line_kind == "comment":
            comment_line = f"# {line_text}" if line_text else "#"
            draw.text((x0, y), comment_line, font=FONT_MONO_22, fill=(38, 120, 77))
        else:
            x = x0
            for token_type, token_value in lex(line_text, lexer):
                if not token_value:
                    continue
                display_value = token_value.replace("\n", " ")
                draw.text((x, y), display_value, font=FONT_MONO_22, fill=token_color(token_type))
                x += draw.textlength(display_value, font=FONT_MONO_22)

            if typing_cursor and line_idx == len(visible_lines) - 1:
                draw.text((x, y), "▌", font=FONT_MONO_22, fill=(20, 20, 20))

    scrollbar_right = cell_box[2] - 10
    scrollbar_left = scrollbar_right - CODE_SCROLLBAR_WIDTH
    scrollbar_top = y0 + 4
    scrollbar_bottom = available_bottom - 4
    draw_scrollbar(
        draw,
        (scrollbar_left, scrollbar_top, scrollbar_right, scrollbar_bottom),
        start_idx=start_idx,
        visible_count=len(visible_lines),
        total_count=len(all_visible_lines),
    )

    if output_items:
        output_top = max(cell_box[1] + 325, y0 + max(len(visible_lines), 1) * line_h + 18)
        output_top = min(output_top, cell_box[3] - 190)

        draw.text((prompt_left, output_top + 12), f"Out[{cell_number}]:", font=FONT_MONO_22, fill=(110, 116, 124))
        output_box = (cell_left, output_top, cell_right, cell_box[3] - 10)
        draw_rounded_rect(draw, output_box, fill=OUTPUT_BG, outline=NOTEBOOK_BORDER, radius=14, width=1)

        text_items = [x for x in output_items if x.get("kind") == "text"]
        image_items = [x for x in output_items if x.get("kind") == "image"]

        if image_items:
            paste_output_image(
                img,
                image_items[0]["image_b64"],
                (output_box[0] + 10, output_box[1] + 10, output_box[2] - 10, output_box[3] - 10),
            )
        elif text_items:
            output_text = "\n".join((item.get("text", "") or "").rstrip() for item in text_items)
            draw_wrapped_text(
                draw,
                output_text,
                output_box[0] + 14,
                output_box[1] + 12,
                FONT_MONO_22,
                (33, 37, 41),
                max_width=(output_box[2] - output_box[0] - 28),
                line_gap=6,
            )

    return img

def save_frame(img, prefix):
    path = FRAME_DIR / f"{prefix}_{uuid.uuid4().hex}.png"
    img.save(path)
    return str(path)

def normalise_audio_path(path):
    return str(Path(path).resolve())

def tts_via_elevenlabs(text, lang_code, prefix="tts"):
    if not ELEVENLABS_API_KEY:
        return None
    out_path = AUDIO_DIR / f"{prefix}_{uuid.uuid4().hex}.mp3"
    url = f"https://api.elevenlabs.io/v1/text-to-speech/{ELEVENLABS_VOICE_ID}"
    headers = {
        "xi-api-key": ELEVENLABS_API_KEY,
        "Content-Type": "application/json",
    }
    payload = {
        "text": text,
        "model_id": ELEVENLABS_MODEL_ID,
        "language_code": lang_code,
    }
    response = requests.post(
        url,
        headers=headers,
        params={"output_format": "mp3_22050_32"},
        json=payload,
        timeout=120,
    )
    if response.status_code >= 400:
        raise RuntimeError(f"ElevenLabs TTS failed: {response.status_code} {response.text[:300]}")
    out_path.write_bytes(response.content)
    return normalise_audio_path(out_path)

def tts_via_gtts(text, lang_code, prefix="tts"):
    out_path = AUDIO_DIR / f"{prefix}_{uuid.uuid4().hex}.mp3"
    gtts_lang = lang_code
    if lang_code in {"zh-CN", "zh-TW"}:
        gtts_lang = "zh-CN"
    tts = gTTS(text=text, lang=gtts_lang)
    tts.save(str(out_path))
    return normalise_audio_path(out_path)

def make_tts_audio(text, lang_code, prefix="tts"):
    text = (text or "").strip()
    if not text:
        return None
    if ELEVENLABS_API_KEY:
        try:
            return tts_via_elevenlabs(text, lang_code, prefix=prefix)
        except Exception as exc:
            print(f"ElevenLabs TTS failed; falling back to gTTS. Reason: {exc}")
    return tts_via_gtts(text, lang_code, prefix=prefix)

def estimate_read_seconds(text):
    words = max(len((text or "").split()), 1)
    return max(MIN_CLIP_SECONDS, (words / 140.0) * 60.0)

def still_clip_from_frame(img, seconds, prefix, audio_path=None):
    frame_path = save_frame(img, prefix)
    clip = ImageClip(frame_path).set_duration(max(seconds, MIN_CLIP_SECONDS))
    if audio_path:
        audio = AudioFileClip(audio_path)
        duration = max(clip.duration, audio.duration)
        clip = clip.set_duration(duration).set_audio(audio)
    return clip

def typing_clips_for_line(existing_display_lines, code_line_text, cell_number, prefix="typing"):
    code_line_text = code_line_text or ""

    if code_line_text == "":
        img = render_notebook_frame(
            cell_kind="code",
            cell_number=cell_number,
            display_lines=existing_display_lines + [{"kind": "code", "text": ""}],
            typing_cursor=False,
        )
        return [still_clip_from_frame(img, max(MIN_CLIP_SECONDS / 2.0, 1.0 / FPS), f"{prefix}_blank")]

    total_chars = len(code_line_text)
    total_seconds = max(MIN_CLIP_SECONDS, total_chars / CODE_TYPING_CPS)
    frame_count = max(2, int(math.ceil(total_seconds * FPS)))
    clips = []

    for frame_idx in range(frame_count):
        ratio = (frame_idx + 1) / frame_count
        visible_chars = max(1, int(total_chars * ratio))
        visible_text = code_line_text[:visible_chars]
        img = render_notebook_frame(
            cell_kind="code",
            cell_number=cell_number,
            display_lines=existing_display_lines,
            typing_current=visible_text,
            typing_cursor=True,
        )
        frame_path = save_frame(img, prefix)
        clip = ImageClip(frame_path).set_duration(1.0 / FPS)
        clips.append(clip)

    return clips

**CELL 6: BUILD MP4**

In [13]:
# =============================================
# Cell 6: Build MP4 from the presentation plan
# =============================================

video_clips = []

for plan_idx, item in enumerate(PRESENTATION_PLAN, start=1):
    if item["cell_type"] == "markdown":
        translated = (item.get("translated_text") or "").strip()
        if translated:
            audio_path = make_tts_audio(translated, TARGET_LANGUAGE_CODE, prefix=f"markdown_{plan_idx}")
            img = render_notebook_frame(
                cell_kind="markdown",
                markdown_text=translated,
            )
            clip = still_clip_from_frame(
                img,
                seconds=estimate_read_seconds(translated) + PAUSE_AFTER_MARKDOWN,
                prefix=f"markdown_{plan_idx}",
                audio_path=audio_path,
            )
            video_clips.append(clip)

    elif item["cell_type"] == "code":
        display_lines = []
        code_cell_number = item.get("code_cell_number", plan_idx)
        events = item.get("events") or []
        outputs = item.get("outputs") or []

        for event_idx, event in enumerate(events, start=1):
            event_kind = event.get("kind")
            event_text = event.get("text", "")

            if event_kind == "script":
                script_audio = make_tts_audio(event_text, TARGET_LANGUAGE_CODE, prefix=f"script_{plan_idx}_{event_idx}")
                img = render_notebook_frame(
                    cell_kind="code",
                    cell_number=code_cell_number,
                    display_lines=display_lines,
                    narration_active=True,
                )
                clip = still_clip_from_frame(
                    img,
                    seconds=estimate_read_seconds(event_text) + SCRIPT_HOLD_SECONDS,
                    prefix=f"script_{plan_idx}_{event_idx}",
                    audio_path=script_audio,
                )
                video_clips.append(clip)

            elif event_kind == "comment":
                display_lines = display_lines + [{"kind": "comment", "text": event_text}]
                hold_seconds = max(MIN_CLIP_SECONDS * 0.7, estimate_read_seconds(event_text) * 0.6) if event_text else (1.0 / FPS)
                img = render_notebook_frame(
                    cell_kind="code",
                    cell_number=code_cell_number,
                    display_lines=display_lines,
                    narration_active=False,
                )
                clip = still_clip_from_frame(
                    img,
                    seconds=hold_seconds + PAUSE_AFTER_COMMENTS,
                    prefix=f"comment_{plan_idx}_{event_idx}",
                    audio_path=None,
                )
                video_clips.append(clip)

            elif event_kind == "code":
                typing_segments = typing_clips_for_line(
                    existing_display_lines=display_lines,
                    code_line_text=event_text,
                    cell_number=code_cell_number,
                    prefix=f"code_{plan_idx}_{event_idx}",
                )
                video_clips.extend(typing_segments)
                display_lines = display_lines + [{"kind": "code", "text": event_text}]

        if outputs:
            output_text_blocks = [x.get("text", "") for x in outputs if x.get("kind") == "text"]
            estimated = estimate_read_seconds("\n".join(output_text_blocks)) if output_text_blocks else MIN_CLIP_SECONDS
            img = render_notebook_frame(
                cell_kind="code",
                cell_number=code_cell_number,
                display_lines=display_lines,
                output_items=outputs,
                narration_active=False,
            )
            clip = still_clip_from_frame(
                img,
                seconds=estimated + PAUSE_AFTER_OUTPUT,
                prefix=f"output_{plan_idx}",
                audio_path=None,
            )
            video_clips.append(clip)

if not video_clips:
    raise RuntimeError("No clips were generated. Check the source notebook content.")

print(f"Generated {len(video_clips)} video clip segments.")


Generated 206 video clip segments.


**CELL 7: EXPORT MP4

In [14]:

# ===================
# Cell 7: Export MP4
# ===================

final_video = concatenate_videoclips(video_clips, method="compose")
final_video.write_videofile(
    str(OUTPUT_MP4_PATH),
    fps=FPS,
    codec="libx264",
    audio_codec="aac",
)

print()
print(f"MP4 export complete: {OUTPUT_MP4_PATH}")

if IN_COLAB:
    try:
        from google.colab import files
        files.download(str(OUTPUT_MP4_PATH))
    except Exception as exc:
        print(f"Automatic download skipped: {exc}")


Moviepy - Building video /content/PeerReview_Prototype_2_0_Output.mp4.
MoviePy - Writing audio in PeerReview_Prototype_2_0_OutputTEMP_MPY_wvf_snd.mp4


MoviePy - Done.
Moviepy - Writing video /content/PeerReview_Prototype_2_0_Output.mp4



Moviepy - Done !
Moviepy - video ready /content/PeerReview_Prototype_2_0_Output.mp4

MP4 export complete: /content/PeerReview_Prototype_2_0_Output.mp4


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Upgrade hooks already left in place

This notebook is organised so you can later add:

- typing speed controls
- optional print / narrate toggles
- voice selection and voice settings
- richer output rendering
- per-cell include / exclude rules
- alternate themes and layouts
